# HMS — full comparison of every model and ensemble

**Single models:** VIPEEGNet (seeds 2024, 2025, 2026) · PRISM (time-reversal TTA) · EfficientNetB0.
**Ensembles** (built here, all with the same rule): VIPEEGNet ×2 and ×3 (equal-weight average of seeds);
VIPEEGNet + PRISM, ×2 + PRISM, ×3 + PRISM (geometric blend, PRISM weight chosen on the other four folds).
**Rows:** the 6,350 EEG rows rated by ≥10 experts, out-of-fold predictions, matched by eeg_id + exact vote counts.
**Two baselines every result is measured against:** VIPEEGNet (seed 2024) — the reproduced published model — and
**VIPEEGNet ×3**, the strongest ensemble without PRISM. An improvement only counts if its patient-level 95% CI excludes 0.

| § | Question |
|---|---|
| 1 | **Overall:** which model fits the experts' votes best, and which improvements are real? |
| 2 | **Seed noise:** how much does VIPEEGNet change from luck alone, on every metric? |
| 3 | **AUROC vs AUPRC:** does the ranking hold where clinical deployment operates — rare classes, high precision? |
| 4 | **False alarms at fixed recall:** to catch 80 / 90 / 95% of cases, how many false alarms does each model raise? |
| 5 | **Calibration:** are the probabilities trustworthy, overall and per class? |
| 6 | **Threshold stability:** does an alarm threshold set on some patients still work on new patients? |
| 7 | **Failure modes:** what kinds of mistakes, how often, and how confidently? |
| 8 | **Soft-label fidelity:** do models fail where experts disagree, and do they reproduce the experts' second choice? |
| 9 | **Heterogeneity:** worst-fold floor, worst patients, and performance by model confidence |
| 10 | **Complementarity:** whose errors overlap — which combinations are worth having? |
| 11 | **Cost vs benefit:** what does each extra model buy? |
| 12 | **Verdict** |

Outputs: `results/comparisons/full_comparison/` (a CSV per section, figures, `report.md` with every answer).

In [ ]:
PROJECT_DIR = '~/hms'
SINGLES = [                                   # [archive tag, display name]
    ['vipeegnet_groupkfold_v1', 'VIPEEGNet (seed 2024)'],
    ['vipeegnet_seed2025_v1',   'VIPEEGNet (seed 2025)'],
    ['vipeegnet_seed2026_v1',   'VIPEEGNet (seed 2026)'],
    ['prism_wf5_tta_v1',        'PRISM'],
    ['effnetb0_native_v1',      'EfficientNetB0'],
]
PARAMS_M = {'VIPEEGNet': 12.94, 'PRISM': 2.96, 'EfficientNetB0': 4.05}   # millions of parameters per member (approx.)
N_BOOT = 1000          # patient resamples for mean-type metrics
N_BOOT_AUC = 200       # patient resamples for AUROC / AUPRC
OUT_NAME = 'full_comparison'

In [ ]:
import os, sys, json, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.stats import spearmanr, norm as norm_dist
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix, f1_score
P = os.path.abspath(os.path.expanduser(PROJECT_DIR))
sys.path.insert(0, f'{P}/results'); os.environ['HMS_RESULTS'] = f'{P}/results'
from load_results import load_oof
CLASS = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']; SZ, LPD, GPD, LRDA, GRDA, OTH = range(6)
OUTD = f'{P}/results/comparisons/{OUT_NAME}'; os.makedirs(OUTD, exist_ok=True)
pid = pd.read_csv(f'{P}/hms-data/train.csv', usecols=['eeg_id', 'patient_id']).groupby('eeg_id').patient_id.first()
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

def kld(t, p): t = np.clip(t, 1e-7, 1); p = np.clip(p, 1e-7, 1); return (t * np.log(t / p)).sum(1)
def norm(p): p = np.clip(p, 1e-7, 1); return p / p.sum(1, keepdims=True)
def geo(a, b, w):
    q = np.exp((1 - w) * np.log(np.clip(a, 1e-7, 1)) + w * np.log(np.clip(b, 1e-7, 1))); return q / q.sum(1, keepdims=True)
def keyed(tag):
    yt, yp, m = load_oof(tag, 'hq')
    v = np.rint(yt * m.votes.values[:, None]).astype(int)
    m['key'] = m.eeg_id.astype(str) + ':' + pd.Series([','.join(map(str, r)) for r in v]).values
    assert m.key.is_unique, f'{tag}: duplicate row keys'
    return yt, norm(yp), m.reset_index(drop=True)

# ---- load and align every available single model ----
base = None; preds, folds, missing = {}, {}, []
for tag, name in SINGLES:
    if not os.path.isdir(f'{P}/results/{tag}'): missing.append(name); continue
    yt_, yp_, m_ = keyed(tag)
    if base is None: base = (yt_, m_); order = np.arange(len(m_))
    else:
        assert set(m_.key) == set(base[1].key), f'{name} was scored on different rows'
        order = pd.Series(np.arange(len(m_)), index=m_.key)[base[1].key].values
        assert np.allclose(yt_[order], base[0], atol=1e-5), f'{name}: labels differ'
    preds[name] = yp_[order]; folds[name] = m_.fold.values[order]
if missing: print('not archived, left out:', missing)
yt, meta = base; meta['patient_id'] = meta.eeg_id.map(pid).values
N = len(yt); Y = yt.argmax(1); OH = (Y[:, None] == np.arange(6)); VOTES = np.rint(yt * meta.votes.values[:, None]).astype(int)
REF = 'VIPEEGNet (seed 2024)'; FV = folds[REF]
SEEDS = [n for n in preds if n.startswith('VIPEEGNet (seed')]

# ---- ensembles: seeds averaged with equal weight; PRISM added by geometric blend, weight chosen on the other folds ----
W = np.round(np.arange(0, 1.001, 0.05), 2); BLEND_W = {}
def lofo_geo(a, b, label):
    out = np.zeros_like(yt); ws = []
    for k in np.unique(FV):
        tr, te = FV != k, FV == k
        w = W[np.argmin([kld(yt[tr], geo(a[tr], b[tr], w)).mean() for w in W])]; out[te] = geo(a[te], b[te], w); ws.append(float(w))
    BLEND_W[label] = ws; return out
MEMBERS = {n: {'VIPEEGNet': 1} if n in SEEDS else {n: 1} for n in preds}
for k in (2, 3):
    if len(SEEDS) >= k:
        preds[f'VIPEEGNet ×{k}'] = norm(sum(preds[s] for s in SEEDS[:k]) / k); MEMBERS[f'VIPEEGNet ×{k}'] = {'VIPEEGNet': k}
if 'PRISM' in preds:
    for src, name, k in ((REF, 'VIPEEGNet + PRISM', 1), ('VIPEEGNet ×2', 'VIPEEGNet ×2 + PRISM', 2), ('VIPEEGNet ×3', 'VIPEEGNet ×3 + PRISM', 3)):
        if src in preds: preds[name] = lofo_geo(preds[src], preds['PRISM'], name); MEMBERS[name] = {'VIPEEGNet': k, 'PRISM': 1}
for n in preds:
    if n not in folds: folds[n] = FV                     # ensembles live on VIPEEGNet's folds
ALL = list(preds)
BASE3 = 'VIPEEGNet ×3' if 'VIPEEGNet ×3' in preds else ('VIPEEGNet ×2' if 'VIPEEGNet ×2' in preds else REF)
TOP = [n for n in ['VIPEEGNet ×3 + PRISM', 'VIPEEGNet ×2 + PRISM', 'VIPEEGNet + PRISM'] if n in preds][:1]
HEAD = [n for n in [REF, 'EfficientNetB0', 'PRISM', BASE3] + TOP if n in preds]
HEAD = list(dict.fromkeys(HEAD))
print('PRISM weight per fold in each blend:', BLEND_W)
print(f'{N:,} rows | models: {ALL}\nheadline set for charts: {HEAD}')

# ---- shared machinery ----
pats, inv = np.unique(meta.patient_id.values, return_inverse=True); NP = len(pats); CNT = np.bincount(inv)
rng = np.random.default_rng(0); BS = rng.integers(0, NP, (N_BOOT, NP))
PAT_ROWS = [np.where(inv == j)[0] for j in range(NP)]
BS_ROWS = [np.concatenate([PAT_ROWS[j] for j in s]) for s in rng.integers(0, NP, (N_BOOT_AUC, NP))]
def boot_mean(x): s = np.bincount(inv, x); return s[BS].sum(1) / CNT[BS].sum(1)
def ci_str(b, f='%.4f'): return (f + ' to ' + f) % (np.percentile(b, 2.5), np.percentile(b, 97.5))
def pval(diff, b): return float(2 * norm_dist.sf(abs(diff) / max(np.std(b, ddof=1), 1e-12)))
K = {n: kld(yt, p) for n, p in preds.items()}
PRED = {n: p.argmax(1) for n, p in preds.items()}
TARGETS = {c: (OH[:, i], (lambda i: lambda p: p[:, i])(i)) for i, c in enumerate(CLASS)}
TARGETS['Harmful (any IIIC) vs Other'] = (Y != OTH, lambda p: 1 - p[:, OTH])

COLS5 = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']      # validated categorical order
COL = dict(zip(HEAD, COLS5)); MUTED = '#a3a29d'
SURF, INK, INK2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF, 'font.size': 10,
                     'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
                     'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False})
def save(fig, name): fig.savefig(f'{OUTD}/{name}', dpi=150); plt.close(fig); display(Image(f'{OUTD}/{name}'))
def legend_below(fig, ax, n): fig.legend(*ax.get_legend_handles_labels(), loc='lower center', ncol=n, fontsize=9)
def dots(ax, table, models, xlabels):
    x = np.arange(len(xlabels)); off = np.linspace(-0.3, 0.3, len(models))
    for o, n in zip(off, models): ax.plot(x + o, table[n].values, 'o', ms=7.5, color=COL[n], mec=SURF, mew=1.8, label=n)
    ax.set_xticks(x, xlabels); ax.tick_params(axis='x', colors=INK, length=0); ax.grid(axis='x', visible=False)
REPORT = [f'# HMS — full comparison ({pd.Timestamp.today().date()})',
          f'{N:,} rows with >=10 votes, {NP:,} patients. Baselines: {REF} and {BASE3}. CIs resample whole patients.']
ANSWERS = []
def mdt(df):
    try: return df.round(4).to_markdown()
    except ImportError: return '```\n' + df.round(4).to_string() + '\n```'
def answer(q, text): ANSWERS.append(f'**{q}** {text}'); print('ANSWER:', text)

## 1. Overall — which model fits the experts' votes best, and which improvements are real?
KLD is the competition metric. Each model is compared with both baselines; a difference is real when its 95% patient CI excludes 0.

In [ ]:
def macro_auc(p, rows=None, f=roc_auc_score):
    r = slice(None) if rows is None else rows
    return np.mean([f(OH[r, c], p[r, c]) for c in range(6)])
rows = []
for n in ALL:
    p = preds[n]; b = boot_mean(K[n]); r = {'model': n, 'members': sum(MEMBERS[n].values()), 'KLD': K[n].mean(), 'KLD 95% CI': ci_str(b)}
    for base_ in (REF, BASE3):
        d = boot_mean(K[n] - K[base_]); diff = (K[n] - K[base_]).mean()
        r[f'vs {base_}'] = diff; r[f'vs {base_} CI'] = ci_str(d); r[f'real vs {base_}'] = (n != base_) and (np.percentile(d, 97.5) < 0 or np.percentile(d, 2.5) > 0)
    r.update({'accuracy %': (PRED[n] == Y).mean() * 100, 'macro-F1 %': f1_score(Y, PRED[n], average='macro') * 100,
              'macro-AUROC': macro_auc(p), 'macro-AUPRC': macro_auc(p, f=average_precision_score), 'Brier': ((p - yt) ** 2).sum(1).mean()})
    rows.append(r)
Q1 = pd.DataFrame(rows).set_index('model'); Q1.to_csv(f'{OUTD}/q1_overall.csv')
display(Q1.round(4))
steps = [(REF, 'VIPEEGNet ×2', 'add a 2nd VIPEEGNet seed'), ('VIPEEGNet ×2', 'VIPEEGNet ×3', 'add a 3rd VIPEEGNet seed'),
         (REF, 'VIPEEGNet + PRISM', 'add PRISM to one VIPEEGNet'), ('VIPEEGNet ×2', 'VIPEEGNet ×2 + PRISM', 'add PRISM to two seeds'),
         ('VIPEEGNet ×3', 'VIPEEGNet ×3 + PRISM', 'add PRISM to three seeds'), ('VIPEEGNet ×3', 'VIPEEGNet ×2 + PRISM', 'PRISM instead of the 3rd seed')]
lad = []
for a, b_, what in steps:
    if a in K and b_ in K:
        d = boot_mean(K[b_] - K[a]); diff = (K[b_] - K[a]).mean()
        lad.append({'step': what, 'from': a, 'to': b_, 'KLD change': diff, '95% CI': ci_str(d), 'p': pval(diff, d),
                    'real?': np.percentile(d, 97.5) < 0 or np.percentile(d, 2.5) > 0})
Q1L = pd.DataFrame(lad).set_index('step'); Q1L.to_csv(f'{OUTD}/q1_improvement_ladder.csv'); display(Q1L.round(4))
best = Q1.KLD.idxmin()
answer('Q1.', f'Lowest KLD: {best} ({Q1.KLD[best]:.4f}) vs {REF} {Q1.KLD[REF]:.4f} and {BASE3} {Q1.KLD[BASE3]:.4f}. '
       f'Real vs {BASE3}: {bool(Q1.loc[best, "real vs " + BASE3])} (difference {Q1.loc[best, "vs " + BASE3]:+.4f}, CI {Q1.loc[best, "vs " + BASE3 + " CI"]}).')
REPORT += ['## 1. Overall', mdt(Q1), '### Improvement ladder (negative = better)', mdt(Q1L)]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.42 * len(ALL) + 1.4)); y = np.arange(len(ALL))[::-1]
CI = {n: np.percentile(boot_mean(K[n]), [2.5, 97.5]) for n in ALL}
med = np.median([K[n].mean() for n in ALL]); far = [n for n in ALL if K[n].mean() > 1.6 * med]   # outliers drawn as arrows
near = [n for n in ALL if n not in far]
lo_ = min(CI[n][0] for n in near); hi_ = max(CI[n][1] for n in near); right = hi_ + (hi_ - lo_) * 0.35
for yi, n in zip(y, ALL):
    c = COL.get(n, MUTED); lo, hi = CI[n]
    if n in far:
        ax.annotate('', xy=(right, yi), xytext=(right - (hi_ - lo_) * 0.12, yi), arrowprops=dict(arrowstyle='-|>', color=c, lw=2))
        ax.text(right - (hi_ - lo_) * 0.14, yi, f'{K[n].mean():.4f} (off scale)', va='center', ha='right', color=INK, fontsize=9)
        continue
    ax.plot([lo, hi], [yi, yi], color=c, lw=2, solid_capstyle='round'); ax.plot(K[n].mean(), yi, 'o', ms=8, color=c, mec=SURF, mew=2)
    ax.text(hi + (hi_ - lo_) * 0.015, yi, f'{K[n].mean():.4f}', va='center', color=INK, fontsize=9, bbox=dict(fc=SURF, ec='none', pad=1))
for base_, ls in ((REF, (0, (3, 3))), (BASE3, (0, (1, 2)))):
    ax.axvline(K[base_].mean(), color=INK2, lw=1, ls=ls)
ax.set_yticks(y, ALL); ax.tick_params(axis='y', colors=INK, length=0); ax.grid(axis='y', visible=False)
ax.set_xlabel(f'KLD, rows with >=10 votes (lower is better) · bars = 95% patient CI\ndashed line = {REF} · dotted line = {BASE3}')
ax.set_title(f'Best: {best} at {K[best].mean():.4f}', loc='left', fontsize=11.5, color=INK)
ax.set_xlim(lo_ - (hi_ - lo_) * 0.05, right + (hi_ - lo_) * 0.02); fig.tight_layout(); save(fig, 'fig01_overall_kld.png')

## 2. Seed noise — how much does VIPEEGNet change from luck alone?

In [ ]:
if len(SEEDS) >= 2:
    mcols = ['KLD', 'accuracy %', 'macro-F1 %', 'macro-AUROC', 'macro-AUPRC', 'Brier']
    s = Q1.loc[SEEDS, mcols]
    Q2 = pd.DataFrame({'mean of seeds': s.mean(), 'SD across seeds': s.std(), 'range': s.max() - s.min(),
                       f'{BASE3} minus {REF}': Q1.loc[BASE3, mcols] - Q1.loc[REF, mcols]})
    if TOP: Q2[f'{TOP[0]} minus {BASE3}'] = Q1.loc[TOP[0], mcols] - Q1.loc[BASE3, mcols]
    Q2['ensemble gain in seed SDs'] = (Q1.loc[TOP[0] if TOP else BASE3, mcols] - s.mean()) / s.std().replace(0, np.nan)
    Q2.to_csv(f'{OUTD}/q2_seed_noise.csv'); display(Q2.round(4))
    answer('Q2.', f'KLD varies by SD {s.KLD.std():.4f} (range {s.KLD.max() - s.KLD.min():.4f}) across {len(SEEDS)} seeds; '
           f'accuracy by SD {s["accuracy %"].std():.2f} points. Differences smaller than about two SDs are within seed luck.')
    REPORT += ['## 2. Seed noise', mdt(Q2)]
else:
    print('only one VIPEEGNet seed archived - section skipped')

## 3. AUROC vs AUPRC — does the ranking hold where deployment operates?
AUROC counts every true negative, so with rare classes (Seizure ≈ 4% of EEGs) it can look excellent while precision is
poor. **AUPRC** averages precision over all recall levels: it directly reflects how many alarms are false, and its chance
level is the class prevalence. **Lift** = AUPRC ÷ prevalence. **pAUROC (FPR ≤ 10%)** looks only at the high-specificity part of the ROC curve.

In [ ]:
rows = []; bootAP = {n: [] for n in ALL}
for n in ALL:
    p = preds[n]
    for t, (y, f) in TARGETS.items():
        s = f(p); ap = average_precision_score(y, s)
        rows.append({'model': n, 'target': t, 'prevalence %': y.mean() * 100, 'AUROC': roc_auc_score(y, s),
                     'pAUROC (FPR<=10%)': roc_auc_score(y, s, max_fpr=0.1), 'AUPRC': ap, 'AUPRC lift': ap / y.mean()})
    for r in BS_ROWS: bootAP[n].append(np.mean([average_precision_score(OH[r, c], p[r, c]) for c in range(6)]))
Q3 = pd.DataFrame(rows)
apci = {n: ci_str(np.array(bootAP[n]), '%.3f') for n in ALL}
Q3.to_csv(f'{OUTD}/q3_auroc_auprc.csv', index=False)
for m_ in ('AUROC', 'AUPRC', 'AUPRC lift', 'pAUROC (FPR<=10%)'):
    print(m_); display(Q3.pivot(index='target', columns='model', values=m_).loc[list(TARGETS), ALL].round(3))
macro = Q3[Q3.target.isin(CLASS)].groupby('model')[['AUROC', 'AUPRC']].mean().loc[ALL]; macro['macro-AUPRC 95% CI'] = pd.Series(apci)
d_ap = np.array(bootAP[TOP[0] if TOP else BASE3]) - np.array(bootAP[BASE3])
gap = Q3.pivot(index='target', columns='model', values='AUROC') - Q3.pivot(index='target', columns='model', values='AUPRC')
rank_flip = [t for t in TARGETS if Q3[Q3.target == t].set_index('model').AUROC.idxmax() != Q3[Q3.target == t].set_index('model').AUPRC.idxmax()]
display(macro.round(4))
rare = Q3[Q3.target == 'Seizure'].set_index('model')
answer('Q3.', f'Seizure (prevalence {rare["prevalence %"].iloc[0]:.1f}%): AUROC {rare.AUROC.min():.3f}-{rare.AUROC.max():.3f} across models, '
       f'but AUPRC only {rare.AUPRC.min():.3f}-{rare.AUPRC.max():.3f}; best AUPRC {rare.AUPRC.idxmax()}. Largest AUROC-AUPRC gap: '
       f'{gap.mean(1).idxmax()}. Targets where the AUPRC winner differs from the AUROC winner: {rank_flip or "none"}.')
REPORT += ['## 3. AUROC vs AUPRC', '### AUPRC', mdt(Q3.pivot(index='target', columns='model', values='AUPRC').loc[list(TARGETS), ALL]),
           '### AUPRC lift over chance', mdt(Q3.pivot(index='target', columns='model', values='AUPRC lift').loc[list(TARGETS), ALL]),
           '### AUROC', mdt(Q3.pivot(index='target', columns='model', values='AUROC').loc[list(TARGETS), ALL]),
           '### Macro averages', mdt(macro)]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13.5, 4.6))
tl = list(TARGETS); short = [t if len(t) < 10 else 'Harmful' for t in tl]
for ax, m_, title in ((axs[0], 'AUROC', 'AUROC: looks excellent everywhere'), (axs[1], 'AUPRC', 'AUPRC: the rare classes are far from solved')):
    piv = Q3.pivot(index='target', columns='model', values=m_).loc[tl]; dots(ax, piv, HEAD, short)
    ax.set_title(title, loc='left', fontsize=11, color=INK); ax.set_ylim(0, 1.02)
prev = Q3[Q3.model == HEAD[0]].set_index('target').loc[tl, 'prevalence %'].values / 100
axs[1].plot(np.arange(len(tl)), prev, '_', ms=26, mew=2, color=INK2, label='chance level (prevalence)')
axs[1].text(len(tl) - 0.6, prev[-1] + 0.03, 'chance', fontsize=8.5, color=INK2, ha='right')
fig.suptitle('Same predictions, two views: AUROC (left) vs AUPRC (right), one-vs-rest per class', x=0.01, ha='left', fontsize=12, color=INK)
legend_below(fig, axs[0], len(HEAD)); fig.tight_layout(rect=(0, 0.07, 1, 1)); save(fig, 'fig02_auroc_vs_auprc.png')
fig, axs = plt.subplots(1, 4, figsize=(16, 4.3), sharey=True)
for ax, t in zip(axs, ['Seizure', 'LRDA', 'GRDA', 'Harmful (any IIIC) vs Other']):
    y, f = TARGETS[t]
    for n in HEAD:
        pr, rc, _ = precision_recall_curve(y, f(preds[n])); ax.plot(rc * 100, pr * 100, lw=1.8, color=COL[n], label=n)
    ax.axvline(90, color=INK2, lw=0.8, ls=(0, (3, 3))); ax.axhline(y.mean() * 100, color=INK2, lw=0.8)
    ax.text(2, y.mean() * 100 + 2, 'chance', fontsize=8, color=INK2)
    ax.set_title(t if len(t) < 10 else 'Harmful vs Other', loc='left', fontsize=10.5, color=INK); ax.set_xlabel('recall % (sensitivity)')
axs[0].set_ylabel('precision % (alarms that are right)'); axs[0].set_ylim(0, 101)
fig.suptitle('Precision-recall curves: how precision collapses as recall rises (dashed = 90% recall)', x=0.01, ha='left', fontsize=12, color=INK)
legend_below(fig, axs[0], len(HEAD)); fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, 'fig03_pr_curves.png')

## 4. False alarms at fixed recall — what does it cost to catch 80 / 90 / 95% of cases?

In [ ]:
RECALLS = [0.80, 0.90, 0.95]
def at_recall(y, s, r):
    fpr, tpr, thr = roc_curve(y, s); i = np.where(tpr >= r)[0][0]
    tp, fp = tpr[i] * y.sum(), fpr[i] * (~y).sum()
    return {'threshold': thr[i], 'precision %': tp / max(tp + fp, 1e-9) * 100, 'false alarms per 100 EEGs': fp / len(y) * 100,
            'false alarms per true detection': fp / max(tp, 1e-9), 'specificity %': (1 - fpr[i]) * 100}
rows = []
for n in ALL:
    for t, (y, f) in TARGETS.items():
        for r in RECALLS: rows.append({'model': n, 'target': t, 'recall': f'{int(r * 100)}%', **at_recall(y, f(preds[n]), r)})
Q4 = pd.DataFrame(rows); Q4.to_csv(f'{OUTD}/q4_false_alarms_at_recall.csv', index=False)
for r in ('90%', '95%'):
    for m_ in ('false alarms per 100 EEGs', 'precision %'):
        print(f'{m_} at {r} recall'); display(Q4[Q4.recall == r].pivot(index='target', columns='model', values=m_).loc[list(TARGETS), ALL].round(1))
fa = Q4[Q4.recall == '90%'].pivot(index='target', columns='model', values='false alarms per 100 EEGs')
answer('Q4.', f'At 90% seizure recall: {fa.loc["Seizure"].idxmin()} raises {fa.loc["Seizure"].min():.1f} false alarms per 100 EEGs vs '
       f'{fa.loc["Seizure", REF]:.1f} for {REF} and {fa.loc["Seizure", "EfficientNetB0"]:.1f} for EfficientNetB0. '
       f'For harmful-vs-Other at 90% recall: {fa.loc["Harmful (any IIIC) vs Other"].min():.1f} ({fa.loc["Harmful (any IIIC) vs Other"].idxmin()}) to '
       f'{fa.loc["Harmful (any IIIC) vs Other"].max():.1f} per 100 EEGs.')
REPORT += ['## 4. False alarms per 100 EEGs at 90% recall', mdt(fa.loc[list(TARGETS), ALL]),
           '### False alarms per true detection at 90% recall', mdt(Q4[Q4.recall == '90%'].pivot(index='target', columns='model', values='false alarms per true detection').loc[list(TARGETS), ALL])]
fig, ax = plt.subplots(figsize=(12, 4.4))
dots(ax, fa.loc[list(TARGETS)], HEAD, [t if len(t) < 10 else 'Harmful' for t in TARGETS])
ax.set_ylabel('false alarms per 100 EEGs'); ax.set_ylim(bottom=0)
ax.set_title('False alarms each model raises to catch 90% of cases (lower is better)', loc='left', fontsize=11.5, color=INK)
legend_below(fig, ax, len(HEAD)); fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, 'fig04_false_alarms_at_90_recall.png')

## 5. Calibration — are the probabilities trustworthy?
- **Class-wise ECE:** predicted probability vs the experts' vote share, per class (the right yardstick for soft-label models).
- **Calibration slope** (per class): vote share regressed on predicted probability. 1 = honest; < 1 = probabilities too extreme; > 1 = too timid.
- **Calibration-in-the-large:** mean predicted probability vs mean vote share per class (does it over- or under-call a class overall?).

In [ ]:
def ece_c(p, c, bins=15):
    idx = np.minimum((p[:, c] * bins).astype(int), bins - 1)
    return np.abs(np.bincount(idx, p[:, c], bins) - np.bincount(idx, yt[:, c], bins)).sum() / N
rows = []
for n in ALL:
    p = preds[n]; r = {'model': n, 'class-wise ECE': np.mean([ece_c(p, c) for c in range(6)]), 'Brier': ((p - yt) ** 2).sum(1).mean()}
    for c in range(6):
        r[f'slope {CLASS[c]}'] = np.polyfit(p[:, c], yt[:, c], 1)[0]
        r[f'over-call {CLASS[c]} (pts)'] = (p[:, c].mean() - yt[:, c].mean()) * 100
    rows.append(r)
Q5 = pd.DataFrame(rows).set_index('model'); Q5.to_csv(f'{OUTD}/q5_calibration.csv')
display(Q5[['class-wise ECE', 'Brier'] + [f'slope {c}' for c in CLASS]].round(3))
print('mean predicted minus mean vote share, percentage points (+ = over-calls the class)')
display(Q5[[f'over-call {c} (pts)' for c in CLASS]].round(2))
answer('Q5.', f'Best calibrated: {Q5["class-wise ECE"].idxmin()} (class-wise ECE {Q5["class-wise ECE"].min():.4f}); worst: '
       f'{Q5["class-wise ECE"].idxmax()} ({Q5["class-wise ECE"].max():.4f}). Seizure slope ranges {Q5["slope Seizure"].min():.2f}-{Q5["slope Seizure"].max():.2f}.')
REPORT += ['## 5. Calibration', mdt(Q5)]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.8)); bins = np.linspace(0, 1, 11)
for ax, (title, getp, gett) in zip(axs, (('Seizure probability vs share of experts voting seizure', lambda p: p[:, SZ], lambda: yt[:, SZ]),
                                         ('All classes pooled', lambda p: p.ravel(), lambda: yt.ravel()))):
    ax.plot([0, 100], [0, 100], color=INK2, lw=1, ls=(0, (3, 3)))
    for n in HEAD:
        pf, tf = getp(preds[n]), gett(); idx = np.minimum(np.digitize(pf, bins) - 1, 9)
        pts = [(pf[idx == i].mean() * 100, tf[idx == i].mean() * 100) for i in range(10) if (idx == i).sum() >= 20]
        ax.plot(*zip(*pts), '-o', lw=2, ms=5.5, color=COL[n], mec=SURF, mew=1.2, label=n)
    ax.set_xlabel('predicted probability %'); ax.set_ylabel('mean expert vote share %'); ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_title(title, loc='left', fontsize=10.5, color=INK)
fig.suptitle('Calibration: on the dashed line, a predicted 30% means 30% of experts voted that way', x=0.01, ha='left', fontsize=12, color=INK)
legend_below(fig, axs[0], len(HEAD)); fig.tight_layout(rect=(0, 0.07, 1, 1)); save(fig, 'fig05_calibration.png')

## 6. Threshold stability — does an alarm threshold drift between patient cohorts?
The five patient folds act as five separate cohorts. **(a) Drift:** the threshold that gives 90% recall in each cohort,
and its spread. **(b) Transfer:** set the threshold on four cohorts, apply it to the fifth — does it still catch 90%, and how many false alarms?

In [ ]:
TT = ['Seizure', 'LRDA', 'Harmful (any IIIC) vs Other']; rows, xfer = [], []
for n in ALL:
    for t in TT:
        y, f = TARGETS[t]; s = f(preds[n]); thr = []
        for k in np.unique(FV):
            te, tr = FV == k, FV != k
            thr.append(at_recall(y[te], s[te], 0.90)['threshold'])
            th = at_recall(y[tr], s[tr], 0.90)['threshold']; a = s[te] >= th
            xfer.append({'model': n, 'target': t, 'cohort': int(k), 'held-out recall %': (a & y[te]).sum() / max(y[te].sum(), 1) * 100,
                         'held-out false alarms per 100 EEGs': (a & ~y[te]).sum() / te.sum() * 100})
        rows.append({'model': n, 'target': t, 'threshold mean': np.mean(thr), 'threshold SD': np.std(thr, ddof=1),
                     'threshold range': np.ptp(thr), 'relative drift (SD/mean) %': np.std(thr, ddof=1) / np.mean(thr) * 100})
Q6a = pd.DataFrame(rows); Q6b = pd.DataFrame(xfer)
Q6s = Q6b.groupby(['target', 'model']).agg(**{'mean held-out recall %': ('held-out recall %', 'mean'), 'worst cohort recall %': ('held-out recall %', 'min'),
                                              'recall SD': ('held-out recall %', 'std'), 'mean false alarms /100': ('held-out false alarms per 100 EEGs', 'mean'),
                                              'worst cohort false alarms /100': ('held-out false alarms per 100 EEGs', 'max')})
Q6a.to_csv(f'{OUTD}/q6_threshold_drift.csv', index=False); Q6b.to_csv(f'{OUTD}/q6_threshold_transfer_per_cohort.csv', index=False); Q6s.to_csv(f'{OUTD}/q6_threshold_transfer.csv')
print('(a) threshold for 90% recall, per cohort'); display(Q6a.set_index(['target', 'model']).round(4))
print('(b) threshold set on 4 cohorts, applied to the 5th'); display(Q6s.round(2))
sz = Q6s.loc['Seizure']
answer('Q6.', f'A seizure threshold set for 90% recall on other patients catches {sz["mean held-out recall %"].min():.1f}-{sz["mean held-out recall %"].max():.1f}% '
       f'on new patients on average; the worst cohort drops to {sz["worst cohort recall %"].min():.1f}% ({sz["worst cohort recall %"].idxmin()}). '
       f'Most stable seizure threshold (lowest relative drift): {Q6a[Q6a.target == "Seizure"].set_index("model")["relative drift (SD/mean) %"].idxmin()}.')
REPORT += ['## 6. Threshold stability', '### (a) Drift of the 90%-recall threshold across cohorts', mdt(Q6a.set_index(['target', 'model'])),
           '### (b) Transfer to new patients', mdt(Q6s)]
fig, axs = plt.subplots(1, len(TT), figsize=(14, 4.4), sharey=True)
for ax, t in zip(axs, TT):
    s = Q6b[Q6b.target == t]; x = np.arange(len(HEAD))
    for i, n in enumerate(HEAD):
        v = s[s.model == n]['held-out recall %'].values
        ax.plot(np.full(len(v), i), v, 'o', ms=6, color=COL[n], mec=SURF, mew=1.2, alpha=0.9)
        ax.plot([i - 0.25, i + 0.25], [v.mean()] * 2, color=COL[n], lw=2.5)
    ax.axhline(90, color=INK2, lw=0.9, ls=(0, (3, 3)))
    ax.set_xticks(x, [h.replace(' (seed 2024)', '\n(seed 2024)').replace(' + PRISM', '\n+ PRISM') for h in HEAD], fontsize=8)
    ax.tick_params(axis='x', colors=INK, length=0); ax.grid(axis='x', visible=False)
    ax.set_title(t if len(t) < 10 else 'Harmful vs Other', loc='left', fontsize=10.5, color=INK)
axs[0].set_ylabel('recall % on the held-out cohort')
fig.suptitle('Threshold set for 90% recall on 4 cohorts, applied to the 5th (dots = cohorts, bar = mean, dashed = target)', x=0.01, ha='left', fontsize=12, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.94)); save(fig, 'fig06_threshold_transfer.png')

## 7. Failure modes — what kinds of mistakes, and how confidently?
Wrong top predictions sorted into clinically different types (mutually exclusive, in this order), per 1,000 EEGs.
**Confident errors** = wrong with ≥ 70% confidence: the mistakes a clinician is least likely to question.

In [ ]:
FAM = {LPD: 'PD', GPD: 'PD', LRDA: 'RDA', GRDA: 'RDA'}
TYPES = ['missed seizure', 'false seizure alarm', 'harmful pattern called Other', 'Other called a harmful pattern',
         'right pattern, wrong side (L vs G)', 'periodic vs rhythmic mix-up']
def etype(t, q):
    if t == q: return None
    if t == SZ: return TYPES[0]
    if q == SZ: return TYPES[1]
    if q == OTH: return TYPES[2]
    if t == OTH: return TYPES[3]
    return TYPES[4] if FAM[t] == FAM[q] else TYPES[5]
rows, pairs = [], []
for n in ALL:
    et = pd.Series([etype(t, q) for t, q in zip(Y, PRED[n])]); conf = preds[n].max(1); wrong = PRED[n] != Y
    r = {'model': n}
    for k in TYPES: r[k] = (et == k).sum() / N * 1000
    r['all errors'] = wrong.mean() * 1000; r['confident errors (conf >= 70%)'] = (wrong & (conf >= 0.7)).mean() * 1000
    r['share of errors that are confident %'] = (wrong & (conf >= 0.7)).sum() / max(wrong.sum(), 1) * 100
    rows.append(r)
    cm = pd.crosstab(pd.Series(Y, name='true').map(dict(enumerate(CLASS))), pd.Series(PRED[n], name='pred').map(dict(enumerate(CLASS))))
    st = cm.stack(); st = st[[a != b for a, b in st.index]].sort_values(ascending=False).head(3)
    pairs.append({'model': n, **{f'#{i + 1} confusion': f'{a} -> {b} ({v})' for i, ((a, b), v) in enumerate(st.items())}})
Q7 = pd.DataFrame(rows).set_index('model'); Q7p = pd.DataFrame(pairs).set_index('model')
Q7.to_csv(f'{OUTD}/q7_failure_modes.csv'); Q7p.to_csv(f'{OUTD}/q7_top_confusions.csv')
print('per 1,000 EEGs'); display(Q7.round(1)); display(Q7p)
answer('Q7.', f'Most common failure for {REF}: "{Q7.loc[REF, TYPES].idxmax()}" ({Q7.loc[REF, TYPES].max():.0f} per 1,000). '
       f'Fewest errors overall: {Q7["all errors"].idxmin()} ({Q7["all errors"].min():.0f}/1,000); fewest confident errors: '
       f'{Q7["confident errors (conf >= 70%)"].idxmin()} ({Q7["confident errors (conf >= 70%)"].min():.1f}/1,000).')
REPORT += ['## 7. Failure modes (per 1,000 EEGs)', mdt(Q7), '### Most frequent confusions (true -> predicted, count)', mdt(Q7p)]
fig, ax = plt.subplots(figsize=(10.5, 4.6)); cats = TYPES + ['confident errors (conf >= 70%)']
yy = np.arange(len(cats))[::-1]; off = np.linspace(-0.3, 0.3, len(HEAD))
for o, n in zip(off, HEAD): ax.plot(Q7.loc[n, cats].values, yy + o, 'o', ms=7.5, color=COL[n], mec=SURF, mew=1.8, label=n)
ax.set_yticks(yy, cats); ax.tick_params(axis='y', colors=INK, length=0); ax.grid(axis='y', visible=False)
ax.set_xlabel('per 1,000 EEGs (lower is better)'); ax.set_xlim(left=0)
ax.set_title('Failure modes: what kind of mistake, how often', loc='left', fontsize=11.5, color=INK)
legend_below(fig, ax, len(HEAD)); fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, 'fig07_failure_modes.png')

## 8. Soft-label fidelity — do models fail on cases experts disagree on?
- Performance by **expert agreement** (share of votes on the majority class).
- **Uncertainty tracking:** Spearman correlation between model entropy and expert vote entropy.
- **Second choice:** on EEGs where a second class got ≥ 25% of votes, does the model's top two equal the experts' top two?
- **Shape match:** mean per-EEG Spearman correlation between the six predicted probabilities and the six vote shares.

In [ ]:
share = yt.max(1)
AB = pd.cut(share, [0, 0.5, 0.75, 0.9999, 1.01], labels=['split (<50%)', 'majority (50-75%)', 'strong (75-<100%)', 'unanimous'])
def H(p): p = np.clip(p, 1e-7, 1); return -(p * np.log(p)).sum(1)
srt = np.argsort(-yt, 1); second = yt[np.arange(N), srt[:, 1]] >= 0.25; et2 = np.sort(srt[:, :2], 1)
rows, lev = [], []
for n in ALL:
    p = preds[n]; mt2 = np.sort(np.argsort(-p, 1)[:, :2], 1)
    rho = np.array([spearmanr(p[i], yt[i])[0] for i in range(N)])
    rows.append({'model': n, 'entropy tracking (Spearman)': spearmanr(H(p), H(yt))[0],
                 'top-2 matches experts\' top-2 % (2nd choice >= 25%)': (mt2[second] == et2[second]).all(1).mean() * 100,
                 'mean per-EEG shape correlation': np.nanmean(rho), 'mean model entropy': H(p).mean()})
    for l in AB.categories:
        m = (AB == l)
        lev.append({'model': n, 'agreement': l, 'n': int(m.sum()), 'KLD': K[n][m].mean(), 'accuracy %': (PRED[n][m] == Y[m]).mean() * 100})
Q8 = pd.DataFrame(rows).set_index('model'); Q8.loc['(experts)', 'mean model entropy'] = H(yt).mean(); Q8l = pd.DataFrame(lev)
Q8.to_csv(f'{OUTD}/q8_soft_label_fidelity.csv'); Q8l.to_csv(f'{OUTD}/q8_by_expert_agreement.csv', index=False)
display(Q8.round(3))
for m_ in ('KLD', 'accuracy %'):
    print(m_, 'by expert agreement'); display(Q8l.pivot_table(index=['agreement', 'n'], columns='model', values=m_, sort=False, observed=True)[ALL].round(4))
spl = Q8l[Q8l.agreement == 'split (<50%)'].set_index('model')
answer('Q8.', f'On split-vote EEGs (n={int(spl.n.iloc[0])}) the best is {spl.KLD.idxmin()} (KLD {spl.KLD.min():.4f}); {REF} {spl.KLD[REF]:.4f}. '
       f'Uncertainty tracks expert disagreement best for {Q8["entropy tracking (Spearman)"].idxmax()} (rho {Q8["entropy tracking (Spearman)"].max():.3f}); '
       f'experts\' top-2 reproduced in up to {Q8.iloc[:, 1].max():.1f}% of two-way cases.')
REPORT += ['## 8. Soft-label fidelity', mdt(Q8), '### KLD by expert agreement',
           mdt(Q8l.pivot_table(index=['agreement', 'n'], columns='model', values='KLD', sort=False, observed=True)[ALL])]
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.5)); levels = list(AB.categories); xx = np.arange(len(levels))
ns = Q8l[Q8l.model == REF].set_index('agreement').loc[levels, 'n'].values
for ax, m_, lab in ((axs[0], 'KLD', 'KLD (lower is better)'), (axs[1], 'accuracy %', 'accuracy vs majority %')):
    for n in HEAD:
        ax.plot(xx, Q8l[Q8l.model == n].set_index('agreement').loc[levels, m_].values, '-o', lw=2, ms=6.5, color=COL[n], mec=SURF, mew=1.5, label=n)
    ax.set_xticks(xx, [f'{l}\nn={k}' for l, k in zip(levels, ns)]); ax.tick_params(axis='x', colors=INK, length=0)
    ax.set_title(lab, loc='left', fontsize=10.5, color=INK); ax.grid(axis='x', visible=False)
fig.suptitle('Performance by how much the experts agreed', x=0.01, ha='left', fontsize=12, color=INK)
legend_below(fig, axs[0], len(HEAD)); fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, 'fig08_by_expert_agreement.png')

## 9. Heterogeneity — worst-fold floor, worst patients, and confidence stratification
- **Fold floor:** each single model on its own training folds; ensembles on VIPEEGNet's folds.
- **Patients:** mean KLD per patient — median, 90th percentile, share above 0.5.
- **Confidence strata:** EEGs grouped by the model's top probability — how many fall in each band, and how good is each band?

In [ ]:
rows = []
for n in ALL:
    fk = pd.Series(K[n]).groupby(folds[n]).mean(); pk = pd.Series(K[n]).groupby(inv).mean()
    rows.append({'model': n, 'fold mean': fk.mean(), 'worst fold': fk.max(), 'best fold': fk.min(), 'fold SD': fk.std(),
                 'patient median': pk.median(), 'patient 90th pct': pk.quantile(0.9), '% patients KLD > 0.5': (pk > 0.5).mean() * 100})
Q9 = pd.DataFrame(rows).set_index('model')
CB = [0, 0.4, 0.6, 0.8, 1.01]; CL = ['<40%', '40-60%', '60-80%', '>=80%']; cs = []
for n in ALL:
    conf = preds[n].max(1); b = pd.cut(conf, CB, right=False, labels=CL)
    for l in CL:
        m = (b == l)
        if m.sum() == 0: continue
        cs.append({'model': n, 'confidence': l, 'share of EEGs %': m.mean() * 100, 'accuracy %': (PRED[n][m] == Y[m]).mean() * 100, 'KLD': K[n][m].mean()})
Q9c = pd.DataFrame(cs)
Q9.to_csv(f'{OUTD}/q9_heterogeneity.csv'); Q9c.to_csv(f'{OUTD}/q9_confidence_strata.csv', index=False)
display(Q9.round(4))
for m_ in ('share of EEGs %', 'accuracy %'):
    print(m_, 'by model confidence'); display(Q9c.pivot(index='confidence', columns='model', values=m_).reindex(CL)[ALL].round(1))
answer('Q9.', f'Best worst-fold floor: {Q9["worst fold"].idxmin()} ({Q9["worst fold"].min():.4f}); {REF} worst fold {Q9.loc[REF, "worst fold"]:.4f}. '
       f'Patients with KLD > 0.5: {Q9["% patients KLD > 0.5"].min():.1f}% ({Q9["% patients KLD > 0.5"].idxmin()}) to {Q9["% patients KLD > 0.5"].max():.1f}%. '
       f'When {REF} is >=80% confident it is right {Q9c[(Q9c.model == REF) & (Q9c.confidence == ">=80%")]["accuracy %"].iloc[0]:.1f}% of the time.')
REPORT += ['## 9. Heterogeneity', mdt(Q9), '### Accuracy % by model confidence',
           mdt(Q9c.pivot(index='confidence', columns='model', values='accuracy %').reindex(CL)[ALL]),
           '### Share of EEGs % by model confidence', mdt(Q9c.pivot(index='confidence', columns='model', values='share of EEGs %').reindex(CL)[ALL])]
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
for i, n in enumerate(HEAD):
    fk = pd.Series(K[n]).groupby(folds[n]).mean().values
    axs[0].plot([i, i], [fk.min(), fk.max()], color=COL[n], lw=2); axs[0].plot(np.full(len(fk), i), fk, 'o', ms=6, color=COL[n], mec=SURF, mew=1.2)
    axs[0].text(i + 0.12, fk.max(), f'{fk.max():.3f}', fontsize=8.5, color=INK, va='center')
axs[0].set_xticks(range(len(HEAD)), [h.replace(' (seed 2024)', '\n(seed 2024)').replace(' + PRISM', '\n+ PRISM') for h in HEAD], fontsize=8.5)
axs[0].tick_params(axis='x', colors=INK, length=0); axs[0].grid(axis='x', visible=False)
axs[0].set_ylabel('KLD per fold'); axs[0].set_title('Fold-to-fold spread (label = worst fold)', loc='left', fontsize=10.5, color=INK)
for n in HEAD:
    s = Q9c[Q9c.model == n].set_index('confidence').reindex(CL)
    axs[1].plot(np.arange(len(CL)), s['accuracy %'], '-o', lw=2, ms=6.5, color=COL[n], mec=SURF, mew=1.5, label=n)
axs[1].set_xticks(range(len(CL)), CL); axs[1].tick_params(axis='x', colors=INK, length=0); axs[1].grid(axis='x', visible=False)
axs[1].set_xlabel("model's confidence in its top prediction"); axs[1].set_ylabel('accuracy %')
axs[1].set_title('Accuracy by confidence band', loc='left', fontsize=10.5, color=INK)
legend_below(fig, axs[1], len(HEAD)); fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, 'fig09_heterogeneity.png')

## 10. Complementarity — whose errors overlap?

In [ ]:
SG = [n for n in ALL if n in dict((nm, 1) for _, nm in SINGLES)]
corr = pd.DataFrame([[np.corrcoef(K[a], K[b])[0, 1] for b in SG] for a in SG], index=SG, columns=SG)
bw = pd.DataFrame([[((PRED[a] != Y) & (PRED[b] != Y)).mean() * 100 for b in SG] for a in SG], index=SG, columns=SG)
corr.to_csv(f'{OUTD}/q10_error_correlation.csv'); bw.to_csv(f'{OUTD}/q10_both_wrong.csv')
print('correlation of per-EEG KLD'); display(corr.round(3)); print('% of EEGs both get wrong'); display(bw.round(1))
off = corr.where(~np.eye(len(SG), dtype=bool))
vs = [s for s in SG if s.startswith('VIPEEGNet')]
seed_r = np.nanmean([corr.loc[a, b] for a in vs for b in vs if a != b]) if len(vs) > 1 else np.nan
answer('Q10.', f'VIPEEGNet seeds correlate with each other at r={seed_r:.3f}; PRISM vs {REF} r={corr.loc["PRISM", REF]:.3f}; '
       f'EfficientNetB0 vs {REF} r={corr.loc["EfficientNetB0", REF]:.3f}. Lower r = more complementary errors.')
REPORT += ['## 10. Complementarity', '### Correlation of per-EEG KLD', mdt(corr), '### % of EEGs both models get wrong', mdt(bw)]
fig, ax = plt.subplots(figsize=(6.8, 5.6))
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#f0efec', '#86b6ef', '#2a78d6', '#104281'])
im = ax.imshow(corr.values, cmap=cmap, vmin=0, vmax=1)
for i in range(len(SG)):
    for j in range(len(SG)): ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=9, color='white' if corr.values[i, j] > 0.6 else INK)
lab = [s.replace('VIPEEGNet ', 'VIPEEGNet\n') for s in SG]
ax.set_xticks(range(len(SG)), lab, fontsize=8.5, rotation=0); ax.set_yticks(range(len(SG)), lab, fontsize=8.5); ax.grid(False); ax.tick_params(length=0)
ax.set_title('Correlation of per-EEG errors (lower = more complementary)', loc='left', fontsize=10.5, color=INK)
fig.tight_layout(); save(fig, 'fig10_error_correlation.png')

## 11. Cost vs benefit — what does each extra model buy?

In [ ]:
rows = []
for n in ALL:
    mem = MEMBERS[n]; params = sum(PARAMS_M.get(k, np.nan) * v for k, v in mem.items()); m = sum(mem.values())
    gain = K[REF].mean() - K[n].mean()
    rows.append({'model': n, 'members': m, 'composition': ' + '.join(f'{v}x {k}' for k, v in mem.items()), 'parameters (M)': params,
                 'KLD': K[n].mean(), f'KLD gain vs {REF}': gain, 'gain per extra member': gain / (m - 1) if m > 1 else np.nan,
                 'gain per extra 10M parameters': gain / ((params - PARAMS_M['VIPEEGNet']) / 10) if params > PARAMS_M['VIPEEGNet'] else np.nan})
Q11 = pd.DataFrame(rows).set_index('model'); Q11.to_csv(f'{OUTD}/q11_cost_benefit.csv'); display(Q11.round(4))
ens = Q11[Q11.members > 1]
if len(ens):
    answer('Q11.', f'Largest gain per extra member: {ens["gain per extra member"].idxmax()} ({ens["gain per extra member"].max():.4f}); '
           f'largest gain per extra 10M parameters: {ens["gain per extra 10M parameters"].idxmax()} ({ens["gain per extra 10M parameters"].max():.4f}).')
REPORT += ['## 11. Cost vs benefit (parameters are approximate, edit PARAMS_M)', mdt(Q11)]

## 12. Verdict
One headline number per question, the best model, and whether that best is **significantly** better than VIPEEGNet ×3
(the strongest ensemble without PRISM) on the metrics where a paired patient CI is available.

In [ ]:
fa90 = Q4[Q4.recall == '90%'].pivot(index='target', columns='model', values='false alarms per 100 EEGs')
xf = Q6s.loc['Seizure']
card = pd.DataFrame({
    'Q1 KLD (lower)': Q1.KLD, 'Q1 accuracy % (higher)': Q1['accuracy %'],
    'Q3 macro-AUPRC (higher)': Q1['macro-AUPRC'], 'Q3 Seizure AUPRC (higher)': Q3[Q3.target == 'Seizure'].set_index('model').AUPRC,
    'Q4 seizure false alarms /100 at 90% recall (lower)': fa90.loc['Seizure'],
    'Q4 harmful false alarms /100 at 90% recall (lower)': fa90.loc['Harmful (any IIIC) vs Other'],
    'Q5 class-wise ECE (lower)': Q5['class-wise ECE'],
    'Q6 worst-cohort seizure recall % (higher)': xf['worst cohort recall %'],
    'Q7 errors /1,000 (lower)': Q7['all errors'], 'Q7 confident errors /1,000 (lower)': Q7['confident errors (conf >= 70%)'],
    'Q8 KLD on split-vote EEGs (lower)': Q8l[Q8l.agreement == 'split (<50%)'].set_index('model').KLD,
    'Q8 entropy tracking (higher)': Q8['entropy tracking (Spearman)'].drop('(experts)'),
    'Q9 worst fold KLD (lower)': Q9['worst fold'], 'Q9 % patients KLD > 0.5 (lower)': Q9['% patients KLD > 0.5'],
}).T[ALL]
low = card.index.str.contains('lower')
card['best'] = [r.idxmin() if l else r.idxmax() for (_, r), l in zip(card[ALL].iterrows(), low)]
card.to_csv(f'{OUTD}/q12_verdict_scorecard.csv')
display(card[HEAD + ['best']].round(4))
wins = card.best.value_counts(); print('wins:', wins.to_dict())
champ = best if best != BASE3 else (TOP[0] if TOP else BASE3)      # the lowest-KLD model, tested against VIPEEGNet x3
sig = {}
dK = boot_mean(K[champ] - K[BASE3]); sig['KLD'] = ((K[champ] - K[BASE3]).mean(), ci_str(dK))
dA = boot_mean((PRED[champ] == Y).astype(float) - (PRED[BASE3] == Y)); sig['accuracy (fraction)'] = (((PRED[champ] == Y).mean() - (PRED[BASE3] == Y).mean()), ci_str(dA))
sig['macro-AUPRC'] = (np.mean(bootAP[champ]) - np.mean(bootAP[BASE3]), ci_str(np.array(bootAP[champ]) - np.array(bootAP[BASE3])))
SIG = pd.DataFrame(sig, index=['difference', '95% CI']).T; display(SIG)
answer('Verdict.', f'Most question wins: {wins.idxmax()} ({wins.max()} of {len(card)}). {champ} vs {BASE3}: KLD {sig["KLD"][0]:+.4f} (CI {sig["KLD"][1]}), '
       f'macro-AUPRC {sig["macro-AUPRC"][0]:+.4f} (CI {sig["macro-AUPRC"][1]}). Single-model ranking by KLD: '
       + ' < '.join(Q1.loc[[n for n in ALL if MEMBERS[n] and sum(MEMBERS[n].values()) == 1]].KLD.sort_values().index) + '.')
REPORT = REPORT[:2] + ['## Answers'] + ['- ' + a for a in ANSWERS] + REPORT[2:] + ['## 12. Verdict scorecard', mdt(card), f'### {champ} vs {BASE3}', mdt(SIG)]

In [ ]:
open(f'{OUTD}/report.md', 'w').write('\n\n'.join(REPORT + ['\n'.join(f'![{f}]({f})' for f in sorted(x for x in os.listdir(OUTD) if x.endswith('.png')))]))
json.dump({'models': ALL, 'members': MEMBERS, 'blend_weights': BLEND_W, 'answers': ANSWERS}, open(f'{OUTD}/summary.json', 'w'), indent=2, default=str)
print('\n'.join(ANSWERS)); print('\nsaved to', OUTD)